In [1]:
%%bash

ls "./fastq/queen_rep1"

queen-rep1-1_S1_L007_R1_001.fastq.gz
queen-rep1-1_S1_L007_R2_001.fastq.gz
queen-rep1-2_S1_L006_R1_001.fastq.gz
queen-rep1-2_S1_L006_R2_001.fastq.gz
queen-rep1-3_S1_L006_R1_001.fastq.gz
queen-rep1-3_S1_L006_R2_001.fastq.gz


Because in the *A. cerana* data, a single directory (e.g., `queen_rep1`) contains multiple Sample_Name (`queen_rep1-1`, `queen_rep1-2`, `queen_rep1-3`), the `cellranger count` step requires the option `--sample=queen_rep1-1,queen_rep1-2,queen_rep1-3`. To handle this, I added a function `get_sample_prefixes` inside `3_cellranger-count.sh`. The function takes a directory as input, extracts all sample names from the `fastq.gz` files directly under that directory, removes duplicates, and joins them with commas. 
The sample name is obtained by taking only the part before the first underscore (`_`). However, files in the Acer directory look like `queen_rep1-1_S1_L007_R1_001.fastq.gz`, so taking the first underscore would only yield `queen`. Therefore, I need to rename files like `queen_rep1-1_S1_L007_R1_001.fastq.gz` to `queen-rep1-1_S1_L007_R1_001.fastq.gz` (replacing the first underscore with a hyphen).

In [11]:
%%bash

for caste_dir in "./fastq"/* ; do
    # echo $caste_dir
    for fq in "$caste_dir"/* ; do
        # echo $fq
        oldname="${fq##*/}"
        newname="${oldname/_/-}"
        mv -n "$fq" "$caste_dir/$newname"
    done
done

In [12]:
%%bash

ls "./fastq/queen_rep1"

queen-rep1-1_S1_L007_R1_001.fastq.gz
queen-rep1-1_S1_L007_R2_001.fastq.gz
queen-rep1-2_S1_L006_R1_001.fastq.gz
queen-rep1-2_S1_L006_R2_001.fastq.gz
queen-rep1-3_S1_L006_R1_001.fastq.gz
queen-rep1-3_S1_L006_R2_001.fastq.gz


In [13]:
%%bash
# verification: 

get_sample_prefixes() {
    local dir="$1"
    if [ ! -d "$dir" ]; then
        echo "Error: '$dir' is not a directory." >&2
        return 1
    fi

    # 遍历目录下所有 .fastq.gz 文件，提取前缀，排序去重，用逗号拼接
    for f in "$dir"/*.fastq.gz; do
        file="${f##*/}"          # 去掉路径，只保留文件名
        echo "${file%%_*}"       # 取第一个 '_' 之前的部分
    done | sort -u | paste -sd,
}

echo $(get_sample_prefixes "./fastq/worker_rep1")
echo $(get_sample_prefixes "./fastq/queen_rep1")
echo $(get_sample_prefixes "./fastq/worker_rep2")
echo $(get_sample_prefixes "./fastq/drone_rep1")
echo $(get_sample_prefixes "./fastq//drone_rep2")

worker-rep1-1,worker-rep1-2,worker-rep1-3,worker-rep1-4,worker-rep1-5
queen-rep1-1,queen-rep1-2,queen-rep1-3
worker-rep2-1,worker-rep2-2,worker-rep2-3,worker-rep2-4,worker-rep2-5,worker-rep2-6,worker-rep2-7
drone-rep1-1,drone-rep1-2,drone-rep1-3,drone-rep1-4,drone-rep1-5,drone-rep1-6,drone-rep1-7,drone-rep1-8
drone-rep2-1,drone-rep2-2,drone-rep2-3,drone-rep2-4,drone-rep2-5,drone-rep2-6,drone-rep2-7,drone-rep2-8
